In [4]:
import sys
from pathlib import Path

import pandas as pd


# ---------------------------------
# Find project root
# ---------------------------------
current_path = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "Divar.csv").exists()
        and (path / "scripts" / "preprocessing_utils.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be detected."
    )

print("Project root:", project_root)


# ---------------------------------
# Add project root to Python path
# ---------------------------------
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# ---------------------------------
# Import central preprocessing
# ---------------------------------
from scripts.preprocessing_utils import (
    run_full_preprocessing_pipeline
)


# ---------------------------------
# File paths
# ---------------------------------
raw_data_path = project_root / "data" / "Divar.csv"

cleaned_data_path = (
    project_root
    / "data"
    / "Divar_cleaned.parquet"
)


# ---------------------------------
# Load raw dataset
# ---------------------------------
raw_df = pd.read_csv(raw_data_path)

print("Raw shape:", raw_df.shape)


# ---------------------------------
# Run full preprocessing pipeline
# ---------------------------------
working_df = raw_df.copy()

pipeline_result = run_full_preprocessing_pipeline(
    working_df
)

# Support both pipeline styles:
# 1. function returns a DataFrame
# 2. function modifies the DataFrame in place
if pipeline_result is None:
    df = working_df
else:
    df = pipeline_result


# ---------------------------------
# Basic validation
# ---------------------------------
print("Cleaned shape:", df.shape)
print("Total columns:", len(df.columns))


# ---------------------------------
# Check mixed object columns
# without modifying them
# ---------------------------------
mixed_object_columns = []

for col in df.select_dtypes(include="object").columns:
    value_types = (
        df[col]
        .dropna()
        .map(type)
        .nunique()
    )

    if value_types > 1:
        mixed_object_columns.append(col)

if mixed_object_columns:
    print(
        "\nMixed-type object columns:",
        mixed_object_columns
    )
else:
    print(
        "\nNo mixed-type object columns detected."
    )


# ---------------------------------
# Save cleaned dataset
# ---------------------------------
df.to_parquet(
    cleaned_data_path,
    engine="pyarrow",
    index=False
)


# ---------------------------------
# Final validation
# ---------------------------------
print(
    "\nPipeline completed successfully."
)

print(
    "Saved to:",
    cleaned_data_path
)

print(
    "File exists:",
    cleaned_data_path.exists()
)

print(
    "File size (MB):",
    round(
        cleaned_data_path.stat().st_size
        / (1024 ** 2),
        2
    )
)

Project root: C:\Users\ASUS\divar-real-estate-project


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17036\2507286811.py:60: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(raw_data_path)


Raw shape: (1000000, 61)
Cleaned shape: (999997, 80)
Total columns: 80

No mixed-type object columns detected.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17036\2507286811.py:96: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:



Pipeline completed successfully.
Saved to: C:\Users\ASUS\divar-real-estate-project\data\Divar_cleaned.parquet
File exists: True
File size (MB): 300.7
